# _First step_

Our objective is to predict fire events using available satellite imagery. Consequently, we must extract and process this data, as it provides the comprehensive spatial coverage required to accommodate the entire state of Mato Grosso

## Data Extraction & Synchronization from Google Earth Engine

This notebook performs the first step of the wildfire prediction pipeline:  
**identifying which geospatial variables are missing locally, requesting their export from Google Earth Engine (GEE), and placing them in the shared data directory.**  

The study area is **Mato Grosso state**, Brazil, and all exports use a **10 km spatial resolution (0.1°)** in EPSG:4326.  
After this notebook completes and all GEE tasks finish, the local folder will contain **all static and dynamic input layers** required by the training blocks.

#### How it works

1. **Initialization** – Authenticate with GEE (`ee.Initialize`) and define the region of interest (Mato Grosso).  
2. **Inventory check** – For each expected file (static or per‑year), the script tests whether it already exists on disk.  
3. **Export queuing** – Missing files are submitted to GEE as batch `Export.image.toDrive` tasks.  
4. **Summary** – A final report tells you how many files were found and how many are still being processed.

**No data is loaded into memory** – only existence checks are performed. The actual download happens asynchronously in Google Drive; from there, `rclone` (or the WSL mount) syncs them to the local `Queimadas_Data` folder.

---

#### Exported variables

#### Static (exported once, no year label)

| Variable | Source | Temporal Resolution | Years Available | Description |
|----------|--------|---------------------|-----------------|-------------|
| Distance to Roads | GRIP4 vector dataset | Static | All (2001–2026) | Euclidean distance to the nearest road (m) – proxies human accessibility and ignition sources. |
| DEM (Elevation) | SRTM GL1 (30 m) | Static | All (2001–2026) | Absolute elevation (m). Controls vegetation type and fire regime. |
| Slope | Derived from SRTM | Static | All (2001–2026) | Terrain slope (degrees). Fire spread rate scales with slope (Rothermel model). |
| Aspect | Derived from SRTM | Static | All (2001–2026) | Slope orientation (degrees). In the Southern Hemisphere, north‑facing slopes receive more insolation → drier fuels. |

#### Dynamic (exported annually, 2001‑present)

| Variable | Source | Temporal Resolution | Years Available | Description / Notes |
|----------|--------|---------------------|-----------------|----------------------|
| Temperature (2 m) | ERA5‑Land daily aggregates | Daily | 2001–2026 | Kelvin; later normalized in training. |
| Precipitation (total) | ERA5‑Land daily aggregates | Daily | 2001–2026 | Total precipitation sum (m). |
| Dewpoint temperature | ERA5‑Land daily aggregates | Daily | 2001–2026 | Used to compute VPD. |
| **Vapor Pressure Deficit (VPD)** | Derived from ERA5 T & Td | Daily | 2001–2026 | Computed as `e_s(T) – e_a(Td)`. Key driver of fuel moisture. See derivation below. |
| Fire Mask (target) | MODIS MOD14A1 (Thermal Anomalies) | Daily | 2001–2026 | Fire mask values 7, 8, 9 → fire event (binarized later). |
| Land Use / Land Cover | MapBiomas Collection 9 | Annual (most recent year ≤2023) | 2001–2026 | For years 2024‑2026, the 2023 classification is repeated. |
| **EVI (Enhanced Vegetation Index)** | MODIS MOD13A2 | 16‑day composites | 2001–2026 | Vegetation greenness; low EVI = senesced, flammable biomass. |
| **SMAP Soil Moisture** | SMAP L3 (SPL3SMP_E) | Daily (gap‑filled) | 2015–2026 | Ascending overpass (6 AM). Encodes multi‑week drought memory. Missing days are filled with the most recent valid observation. |
| Sentinel‑5P CO | TROPOMI L3 offline | Daily | 2018–2026 | Column CO density – proxy for biomass burning emissions. |


> **VPD derivation:**  
> ERA5 provides temperature (T) and dewpoint (Td) in Kelvin. The script converts them to °C, then:  
> `e_s = 0.6108 * exp(17.27 * T_c / (T_c + 237.3))`  
> `e_a = 0.6108 * exp(17.27 * Td_c / (Td_c + 237.3))`  
> `VPD = max(e_s − e_a, 0)`  

> **SMAP gap filling:**  
> Missing days (clouds, sensor gaps) are filled with the most recent valid observation using a JavaScript function in GEE.  

---

#### Output

All GeoTIFFs are saved in the `Queimadas_Data` folder (mounted from Google Drive).  
The file naming convention is:

- Static: `Distance_to_Roads.tif`, `Topography_DEM.tif`, `Topography_Slope.tif`, `Topography_Aspect.tif`
- Dynamic: `ERA5_Temp_{year}.tif`, `ERA5_VPD_{year}.tif`, `MODIS_EVI_{year}.tif`, etc.

---

**After this notebook finishes and all GEE tasks complete (monitored via the GEE Task Manager), the data is ready for Block 2 (Model Training).**

In [ ]:
import ee
import os
from datetime import datetime

# =====================================================================
# BLOCK 1: MULTI-VARIABLE DATA EXTRACTION (GEE TO LOCAL SYNC)
# =====================================================================
# Input channels after this script completes:
#   Existing : Temperature, Precipitation, Dewpoint, LULC, Road Distance, CO
#   New      : VPD (derived), DEM, Slope, Aspect, SMAP Soil Moisture, MODIS EVI
# =====================================================================
ee.Initialize(project='indigo-syntax-420618')

data_dir = "/home/daves/google_drive/Pessoal/Notebooks/Queimadas/Queimadas_Data"
os.makedirs(data_dir, exist_ok=True)

mt_state = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq('ADM1_NAME', 'Mato Grosso'))
roi_bounds = mt_state.geometry().bounds()
region_coords = roi_bounds.getInfo()['coordinates']

current_year = datetime.now().year
years_to_export = range(2001, current_year + 1)

# Common export kwargs to avoid repetition
def export_image(image, description, file_prefix, scale=10000):
    ee.batch.Export.image.toDrive(
        image=image,
        description=description,
        folder='Queimadas_Data',
        fileNamePrefix=file_prefix,
        region=region_coords,
        scale=scale,
        crs='EPSG:4326',
        maxPixels=1e13
    ).start()

print(f"Scanning local WSL directory: {data_dir}\n")

missing_files = 0
found_files = 0


# =====================================================================
# STATIC VARIABLES (Exported once, year-independent)
# =====================================================================

# ---------------------------------------------------------------------
# 1a. STATIC: DISTANCE TO ROADS (GRIP4)  [existing]
# ---------------------------------------------------------------------
f_roads = os.path.join(data_dir, 'Distance_to_Roads.tif')
if not os.path.exists(f_roads):
    print("  [MISSING] Queueing Distance to Roads export...")
    roads = ee.FeatureCollection("projects/sat-io/open-datasets/GRIP4/Central-South-America").filterBounds(roi_bounds)
    distance_to_roads = roads.distance(searchRadius=50000, maxError=100)
    export_image(distance_to_roads, 'Distance_to_Roads', 'Distance_to_Roads')
    missing_files += 1
else:
    print("  [FOUND] Distance to Roads.")
    found_files += 1

# ---------------------------------------------------------------------
# 1b. STATIC: DEM, SLOPE, ASPECT  [new — SRTM 30 m, resampled to 10 km]
#
# Physical rationale:
#   - DEM    : Absolute elevation drives vegetation type and fire frequency.
#   - Slope  : Fire spread rate scales quadratically with slope (Rothermel model).
#   - Aspect : Solar insolation drives differential drying; in the Southern
#              Hemisphere north-facing slopes receive more sun → drier fuels.
# ---------------------------------------------------------------------
f_dem    = os.path.join(data_dir, 'Topography_DEM.tif')
f_slope  = os.path.join(data_dir, 'Topography_Slope.tif')
f_aspect = os.path.join(data_dir, 'Topography_Aspect.tif')

srtm = ee.Image("USGS/SRTMGL1_003")

if not os.path.exists(f_dem):
    print("  [MISSING] Queueing DEM export...")
    export_image(srtm.select('elevation'), 'Topography_DEM', 'Topography_DEM')
    missing_files += 1
else:
    print("  [FOUND] DEM.")
    found_files += 1

if not os.path.exists(f_slope):
    print("  [MISSING] Queueing Slope export...")
    slope = ee.Terrain.slope(srtm)
    export_image(slope, 'Topography_Slope', 'Topography_Slope')
    missing_files += 1
else:
    print("  [FOUND] Slope.")
    found_files += 1

if not os.path.exists(f_aspect):
    print("  [MISSING] Queueing Aspect export...")
    aspect = ee.Terrain.aspect(srtm)
    export_image(aspect, 'Topography_Aspect', 'Topography_Aspect')
    missing_files += 1
else:
    print("  [FOUND] Aspect.")
    found_files += 1


# =====================================================================
# DYNAMIC VARIABLES (Exported per year)
# =====================================================================

for year in years_to_export:
    print(f"\n--- Checking Year {year} ---")
    start_date = f'{year}-01-01'
    end_date   = f'{year+1}-01-01'

    # ------------------------------------------------------------------
    # 2. ERA5 METEOROLOGY  [existing]
    # ------------------------------------------------------------------
    f_temp   = os.path.join(data_dir, f'ERA5_Temp_{year}.tif')
    f_precip = os.path.join(data_dir, f'ERA5_Precip_{year}.tif')
    f_dew    = os.path.join(data_dir, f'ERA5_Dewpoint_{year}.tif')

    era5 = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
              .filterBounds(roi_bounds)
              .filterDate(start_date, end_date))

    if not os.path.exists(f_temp):
        print(f"  [MISSING] Queueing Temperature for {year}...")
        export_image(era5.select('temperature_2m').toBands(), f'ERA5_Temp_{year}', f'ERA5_Temp_{year}')
        missing_files += 1
    else:
        print(f"  [FOUND] Temperature for {year}.")
        found_files += 1

    if not os.path.exists(f_precip):
        print(f"  [MISSING] Queueing Precipitation for {year}...")
        export_image(era5.select('total_precipitation_sum').toBands(), f'ERA5_Precip_{year}', f'ERA5_Precip_{year}')
        missing_files += 1
    else:
        print(f"  [FOUND] Precipitation for {year}.")
        found_files += 1

    if not os.path.exists(f_dew):
        print(f"  [MISSING] Queueing Dewpoint for {year}...")
        export_image(era5.select('dewpoint_temperature_2m').toBands(), f'ERA5_Dewpoint_{year}', f'ERA5_Dewpoint_{year}')
        missing_files += 1
    else:
        print(f"  [FOUND] Dewpoint for {year}.")
        found_files += 1

    # ------------------------------------------------------------------
    # 3. VAPOR PRESSURE DEFICIT (VPD)  [new — derived from ERA5 T & Td]
    #
    # VPD = e_s(T) - e_a(Td)
    #   e_s(T)  = 0.6108 * exp(17.27 * T_C  / (T_C  + 237.3))  [kPa]
    #   e_a(Td) = 0.6108 * exp(17.27 * Td_C / (Td_C + 237.3))  [kPa]
    # where T_C and Td_C are °C.  ERA5 stores values in Kelvin.
    #
    # Physical rationale: VPD is the primary atmospheric driver of fuel
    # moisture loss. It integrates T and Td non-linearly and is used
    # operationally in the Canadian Fire Weather Index and US NFDRS.
    # ------------------------------------------------------------------
    f_vpd = os.path.join(data_dir, f'ERA5_VPD_{year}.tif')

    if not os.path.exists(f_vpd):
        print(f"  [MISSING] Queueing VPD for {year}...")

        def compute_daily_vpd(image):
            t_k  = image.select('temperature_2m')
            td_k = image.select('dewpoint_temperature_2m')
            t_c  = t_k.subtract(273.15)
            td_c = td_k.subtract(273.15)

            # Saturation vapor pressure at T
            es = t_c.divide(t_c.add(237.3)).multiply(17.27).exp().multiply(0.6108)
            # Actual vapor pressure at Td
            ea = td_c.divide(td_c.add(237.3)).multiply(17.27).exp().multiply(0.6108)

            vpd = es.subtract(ea).max(ee.Image.constant(0)).rename('VPD')
            return vpd.copyProperties(image, ['system:time_start'])

        vpd_col = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
                     .filterBounds(roi_bounds)
                     .filterDate(start_date, end_date)
                     .select(['temperature_2m', 'dewpoint_temperature_2m'])
                     .map(compute_daily_vpd))

        export_image(vpd_col.toBands(), f'ERA5_VPD_{year}', f'ERA5_VPD_{year}')
        missing_files += 1
    else:
        print(f"  [FOUND] VPD for {year}.")
        found_files += 1

    # ------------------------------------------------------------------
    # 4. FIRE MASK — MODIS MOD14A1  [existing, target variable]
    # ------------------------------------------------------------------
    f_fire = os.path.join(data_dir, f'MODIS_Fire_{year}.tif')
    if not os.path.exists(f_fire):
        print(f"  [MISSING] Queueing Fire Data for {year}...")
        col = (ee.ImageCollection("MODIS/061/MOD14A1")
                 .filterBounds(roi_bounds)
                 .filterDate(start_date, end_date)
                 .select('FireMask'))
        export_image(col.toBands(), f'MODIS_Fire_{year}', f'MODIS_Fire_{year}')
        missing_files += 1
    else:
        print(f"  [FOUND] Fire Data for {year}.")
        found_files += 1

    # ------------------------------------------------------------------
    # 5. LAND COVER — MapBiomas Collection 9  [existing]
    # ------------------------------------------------------------------
    f_lulc = os.path.join(data_dir, f'MapBiomas_LULC_{year}.tif')
    if not os.path.exists(f_lulc):
        print(f"  [MISSING] Queueing Land Cover for {year}...")
        mb_year = min(year, 2023)
        mb_band = f'classification_{mb_year}'
        mapbiomas = ee.Image("projects/mapbiomas-public/assets/brazil/lulc/collection9/mapbiomas_collection90_integration_v1").select(mb_band)
        export_image(mapbiomas, f'MapBiomas_LULC_{year}', f'MapBiomas_LULC_{year}')
        missing_files += 1
    else:
        print(f"  [FOUND] Land Cover for {year}.")
        found_files += 1

    # ------------------------------------------------------------------
    # 6. EVI (Enhanced Vegetation Index) — MODIS MOD13A2 16-day  [new]
    #
    # Physical rationale: EVI measures vegetation physiological stress.
    # Low EVI → senesced, dry biomass = high flammability.
    # High EVI → green canopy = low flammability.
    # EVI is preferred over NDVI in tropical dense-canopy regions (Cerrado
    # / Amazon transition) because it does not saturate at high LAI and
    # it corrects for atmospheric aerosol contamination from smoke.
    #
    # Strategy: We export each 16-day composite as its own band (up to
    # ~23 bands per year). The model DataLoader handles temporal alignment
    # against the daily ERA5 stack by repeating each composite value for
    # the 16 days it covers.
    # ------------------------------------------------------------------
    f_evi = os.path.join(data_dir, f'MODIS_EVI_{year}.tif')
    if not os.path.exists(f_evi):
        print(f"  [MISSING] Queueing MODIS EVI for {year}...")
        evi_col = (ee.ImageCollection("MODIS/061/MOD13A2")
                     .filterBounds(roi_bounds)
                     .filterDate(start_date, end_date)
                     .select('EVI'))
        export_image(evi_col.toBands(), f'MODIS_EVI_{year}', f'MODIS_EVI_{year}', scale=10000)
        missing_files += 1
    else:
        print(f"  [FOUND] MODIS EVI for {year}.")
        found_files += 1

    # ------------------------------------------------------------------
    # 7. SOIL MOISTURE — SMAP L3 (available from April 2015)  [new]
    #
    # Physical rationale: Soil moisture encodes multi-week drought memory
    # that single-day VPD cannot capture. A region may show high VPD on
    # any given day, but if soils remain saturated (e.g., early dry season)
    # ignition probability stays low. SMAP closes this lag gap.
    #
    # Band used: 'soil_moisture_am' — 6 AM ascending overpass, highest
    # quality flag in humid tropical conditions.
    # Available: ~2015-04-01 onwards.
    # ------------------------------------------------------------------
    f_smap = os.path.join(data_dir, f'SMAP_SoilMoisture_{year}.tif')
    smap_start_year = 2015

    if year >= smap_start_year:
        if not os.path.exists(f_smap):
            print(f"  [MISSING] Queueing SMAP Soil Moisture for {year}...")

            # SMAP has data gaps; fill them with the most-recent valid obs
            def fill_smap_gap(day_offset):
                d = ee.Date(start_date).advance(ee.Number(day_offset), 'day')
                daily = (ee.ImageCollection("NASA/SMAP/SPL3SMP_E/006")
                           .filterBounds(roi_bounds)
                           .filterDate(d, d.advance(1, 'day'))
                           .select('soil_moisture_am'))
                filled = ee.Image(ee.Algorithms.If(
                    daily.size().gt(0),
                    daily.mean(),
                    ee.Image.constant(-9999)
                )).rename('soil_moisture_am').toFloat()
                return filled.set('system:time_start', d.millis())

            start_ee = ee.Date(start_date)
            end_ee   = ee.Date(end_date)
            n_days   = end_ee.difference(start_ee, 'day')
            days_seq = ee.List.sequence(0, n_days.subtract(1))
            smap_col = ee.ImageCollection.fromImages(days_seq.map(fill_smap_gap))

            export_image(smap_col.toBands(), f'SMAP_SoilMoisture_{year}', f'SMAP_SoilMoisture_{year}')
            missing_files += 1
        else:
            print(f"  [FOUND] SMAP Soil Moisture for {year}.")
            found_files += 1
    else:
        print(f"  [SKIPPED] SMAP Soil Moisture (not available before {smap_start_year}).")

    # ------------------------------------------------------------------
    # 8. SENTINEL-5P CO — Proxy for biomass burning emissions  [existing]
    # ------------------------------------------------------------------
    f_co = os.path.join(data_dir, f'Sentinel5P_CO_{year}.tif')
    if year >= 2018:
        if not os.path.exists(f_co):
            print(f"  [MISSING] Queueing Sentinel-5P CO for {year}...")

            def get_daily_co(day_offset):
                d = ee.Date(start_date).advance(ee.Number(day_offset), 'day')
                daily_col = (ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_CO")
                               .filterBounds(roi_bounds)
                               .filterDate(d, d.advance(1, 'day'))
                               .select('CO_column_number_density'))
                daily_img = ee.Image(ee.Algorithms.If(
                    daily_col.size().gt(0),
                    daily_col.mean(),
                    ee.Image.constant(0)
                )).rename('CO').toFloat()
                return daily_img.set('system:time_start', d.millis())

            start_ee = ee.Date(start_date)
            end_ee   = ee.Date(end_date)
            n_days   = end_ee.difference(start_ee, 'day')
            days_seq = ee.List.sequence(0, n_days.subtract(1))
            co_col   = ee.ImageCollection.fromImages(days_seq.map(get_daily_co))

            export_image(co_col.toBands(), f'Sentinel5P_CO_{year}', f'Sentinel5P_CO_{year}')
            missing_files += 1
        else:
            print(f"  [FOUND] Sentinel-5P CO for {year}.")
            found_files += 1
    else:
        print(f"  [SKIPPED] Sentinel-5P CO (not available before 2018).")


# =====================================================================
# FINAL SUMMARY REPORT
# =====================================================================
print("\n" + "=" * 60)
print("VARIABLE INVENTORY (full model):")
print("  Static  : Roads Distance, DEM, Slope, Aspect")
print("  Dynamic : Temperature, Precipitation, Dewpoint, VPD,")
print("            MODIS EVI (16-day), SMAP Soil Moisture,")
print("            MODIS Fire (target), MapBiomas LULC,")
print("            Sentinel-5P CO (2018+), SMAP SM (2015+)")
print("-" * 60)
if missing_files == 0:
    print(f"[SUCCESS] ALL {found_files} REQUIRED FILES FOUND LOCALLY!")
    print("No new downloads were queued to Google Earth Engine.")
    print("You are ready to proceed to Block 2 (Training).")
else:
    print(f"[STATUS] Found {found_files} files locally.")
    print(f"[ACTION] Queued {missing_files} missing files to Earth Engine.")
    print("Monitor GEE Task Manager and wait for Drive sync before Block 2.")
print("=" * 60 + "\n")